# MV Validation — Overlapping-Window Group-Aware Label Split (Issue 3B)

This notebook implements the relatively less restrictive option in issue 3B.

> For the same `login_id + acct_nbr`, only samples whose 90-day windows overlap must stay in one split. Non-overlapping windows from the same employee-account pair may be assigned to different splits.

Overlap is transitive. If window A overlaps B and B overlaps C, all three windows form one indivisible overlap group even when A does not directly overlap C.

The notebook follows the issue 3A notebook: it recovers the original split targets, performs several randomized greedy allocations, validates the hard constraint, and compares label balance. It does not retrain a model or overwrite the source table.

In [ ]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

## 1. Configuration

The permanently saved LightGBM feature table is sufficient because it contains the employee, account, label, original split, and 90-day window boundaries.

In [ ]:
FEATURE_TABLE_PATH = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'ins_us_nms/v1/output/insider_us_nms_features_table_v2'
)

EMPLOYEE_COL = 'login_id'
ACCOUNT_COL = 'acct_nbr'
SAMPLE_DATE_COL = 'date'
WINDOW_START_COL = 'lookback_window_start'
WINDOW_END_COL = 'fraud_date'
TARGET_COL = 'label'
ORIGINAL_SPLIT_COL = 'label_split'
NEW_SPLIT_COL = 'new_label_split'
OVERLAP_GROUP_COL = 'overlap_group_id'

SPLIT_ORDER = ['train', 'val', 'test']
RANDOM_SEED = 42
N_ATTEMPTS = 5

ROW_WEIGHT = 1.0
POSITIVE_WEIGHT = 3.0
GROUP_WEIGHT = 0.25
OVERSHOOT_WEIGHT = 5.0
EPSILON = 1e-12

# Optional assignment output. The source feature table is never overwritten.
WRITE_OUTPUT = False
OUTPUT_PATH = (
    'abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/'
    'mrm/ins_us_nms/output/mv_employee_account_overlap_split_assignment_v1'
)

## 2. Read the permanent feature table

In [ ]:
features_df = (
    spark.read.format('delta').load(FEATURE_TABLE_PATH)
    .select(
        F.upper(F.col(EMPLOYEE_COL)).alias(EMPLOYEE_COL),
        F.col(ACCOUNT_COL).cast('string').alias(ACCOUNT_COL),
        F.to_date(SAMPLE_DATE_COL).alias(SAMPLE_DATE_COL),
        F.to_date(WINDOW_START_COL).alias(WINDOW_START_COL),
        F.to_date(WINDOW_END_COL).alias(WINDOW_END_COL),
        F.col(TARGET_COL).cast('int').alias(TARGET_COL),
        F.col(ORIGINAL_SPLIT_COL),
    )
    .cache()
)

required_cols = [
    EMPLOYEE_COL, ACCOUNT_COL, SAMPLE_DATE_COL, WINDOW_START_COL,
    WINDOW_END_COL, TARGET_COL, ORIGINAL_SPLIT_COL,
]
invalid_input_count = features_df.filter(
    (F.greatest(*[F.col(c).isNull().cast('int') for c in required_cols]) == 1)
    | (F.col(WINDOW_START_COL) > F.col(WINDOW_END_COL))
    | (~F.col(TARGET_COL).isin(0, 1))
    | (~F.col(ORIGINAL_SPLIT_COL).isin(SPLIT_ORDER))
).count()
assert invalid_input_count == 0, f'Found {invalid_input_count} rows with invalid keys, labels, splits, or dates'

total_rows = features_df.count()
total_positives = features_df.agg(F.sum(TARGET_COL).alias('n')).first()['n']
assert total_rows > 0 and total_positives > 0

print('Total rows:', total_rows)
print('Total positives:', total_positives)
display(features_df.limit(10))

## 3. Build transitive overlap groups

Within each employee-account pair, windows are ordered by start date. A new group begins only when the next start date is later than the maximum end date of every preceding window. Because the intervals are inclusive, equal boundary dates count as overlap.

In [ ]:
order_cols = [WINDOW_START_COL, WINDOW_END_COL, SAMPLE_DATE_COL]
previous_rows = (
    Window.partitionBy(EMPLOYEE_COL, ACCOUNT_COL)
    .orderBy(*order_cols)
    .rowsBetween(Window.unboundedPreceding, -1)
)
cumulative_rows = (
    Window.partitionBy(EMPLOYEE_COL, ACCOUNT_COL)
    .orderBy(*order_cols)
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

features_grouped = (
    features_df
    .withColumn('_previous_max_end', F.max(WINDOW_END_COL).over(previous_rows))
    .withColumn(
        '_new_overlap_group',
        F.when(F.col('_previous_max_end').isNull(), 1)
        .when(F.col(WINDOW_START_COL) > F.col('_previous_max_end'), 1)
        .otherwise(0),
    )
    .withColumn(
        'overlap_component',
        F.sum('_new_overlap_group').over(cumulative_rows).cast('long'),
    )
    .withColumn(
        OVERLAP_GROUP_COL,
        F.sha2(
            F.concat_ws(
                '||',
                F.col(EMPLOYEE_COL),
                F.col(ACCOUNT_COL),
                F.col('overlap_component').cast('string'),
            ),
            256,
        ),
    )
    .drop('_previous_max_end', '_new_overlap_group')
    .cache()
)

overlap_group_summary_spk = (
    features_grouped
    .groupBy(OVERLAP_GROUP_COL)
    .agg(
        F.first(EMPLOYEE_COL).alias(EMPLOYEE_COL),
        F.first(ACCOUNT_COL).alias(ACCOUNT_COL),
        F.min(WINDOW_START_COL).alias('group_window_start'),
        F.max(WINDOW_END_COL).alias('group_window_end'),
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
    )
    .withColumn('negative_count', F.col('row_count') - F.col('positive_count'))
)

print('Number of overlap groups:', overlap_group_summary_spk.count())
display(overlap_group_summary_spk.orderBy(F.desc('row_count')).limit(20))

## 4. Recover the original split targets and current overlap leakage

The original row and positive-label shares become the allocation targets. The current leakage table reports overlap groups that presently cross train, validation, or test.

In [ ]:
original_split_summary = (
    features_grouped
    .groupBy(ORIGINAL_SPLIT_COL)
    .agg(
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
        F.avg(TARGET_COL).alias('positive_rate'),
        F.countDistinct(OVERLAP_GROUP_COL).alias('group_count'),
    )
    .withColumn('row_share', F.col('row_count') / F.lit(total_rows))
    .withColumn('positive_share', F.col('positive_count') / F.lit(total_positives))
)

current_group_split_check = (
    features_grouped
    .groupBy(OVERLAP_GROUP_COL)
    .agg(
        F.count('*').alias('row_count'),
        F.sum(TARGET_COL).alias('positive_count'),
        F.countDistinct(ORIGINAL_SPLIT_COL).alias('number_of_splits'),
    )
)

current_leakage_summary = current_group_split_check.agg(
    F.count('*').alias('total_overlap_groups'),
    F.sum((F.col('number_of_splits') > 1).cast('long')).alias('groups_crossing_splits'),
    F.sum(F.when(F.col('number_of_splits') > 1, F.col('row_count')).otherwise(0)).alias('rows_in_crossing_groups'),
    F.sum(F.when(F.col('number_of_splits') > 1, F.col('positive_count')).otherwise(0)).alias('positives_in_crossing_groups'),
)

display(original_split_summary)
display(current_leakage_summary)

## 5. Allocate complete overlap groups

This is the same greedy idea used in the issue 3A notebook, but the indivisible unit is an overlap group rather than an entire employee-account pair. Positive-heavy and large groups are allocated first. Several randomized tie-breaking attempts are tested, and the lowest final objective is retained.

In [ ]:
group_pd = overlap_group_summary_spk.toPandas()
for c in ['row_count', 'positive_count', 'negative_count']:
    group_pd[c] = group_pd[c].astype(np.int64)

original_target_pd = (
    original_split_summary
    .select(
        F.col(ORIGINAL_SPLIT_COL).alias('split'),
        'row_count', 'positive_count', 'group_count', 'row_share', 'positive_share',
    )
    .toPandas()
)
original_target_pd = (
    original_target_pd[original_target_pd['split'].isin(SPLIT_ORDER)]
    .set_index('split').loc[SPLIT_ORDER].reset_index()
)

total_groups = len(group_pd)
original_group_total = original_target_pd['group_count'].sum()
original_target_pd['group_share'] = original_target_pd['group_count'] / original_group_total

targets = {
    row['split']: {
        'rows': float(row['row_share'] * total_rows),
        'positives': float(row['positive_share'] * total_positives),
        'groups': float(row['group_share'] * total_groups),
    }
    for _, row in original_target_pd.iterrows()
}

display(original_target_pd)
print('Allocation units:', total_groups)
print('Targets:', targets)

In [ ]:
def normalized_error(actual, target):
    return abs(actual - target) / max(target, EPSILON)


def normalized_overshoot(actual, target):
    return max(actual - target, 0.0) / max(target, EPSILON)


def candidate_score(rows, positives, groups, target):
    ordinary_error = (
        ROW_WEIGHT * normalized_error(rows, target['rows'])
        + POSITIVE_WEIGHT * normalized_error(positives, target['positives'])
        + GROUP_WEIGHT * normalized_error(groups, target['groups'])
    )
    overshoot = OVERSHOOT_WEIGHT * (
        normalized_overshoot(rows, target['rows'])
        + normalized_overshoot(positives, target['positives'])
        + normalized_overshoot(groups, target['groups'])
    )
    return ordinary_error + overshoot


def allocate_groups(group_data, random_seed):
    work = group_data.copy()
    rng = np.random.default_rng(random_seed)
    work['_tie_breaker'] = rng.random(len(work))
    work = work.sort_values(
        ['positive_count', 'row_count', '_tie_breaker'],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    current = {
        split: {'rows': 0.0, 'positives': 0.0, 'groups': 0.0}
        for split in SPLIT_ORDER
    }
    assignments = []

    for row in work.itertuples(index=False):
        scores = {}
        for split in SPLIT_ORDER:
            scores[split] = candidate_score(
                current[split]['rows'] + row.row_count,
                current[split]['positives'] + row.positive_count,
                current[split]['groups'] + 1,
                targets[split],
            )
        best_split = min(scores, key=scores.get)
        assignments.append(best_split)
        current[best_split]['rows'] += row.row_count
        current[best_split]['positives'] += row.positive_count
        current[best_split]['groups'] += 1

    work[NEW_SPLIT_COL] = assignments
    final_objective = sum(
        candidate_score(
            current[s]['rows'], current[s]['positives'], current[s]['groups'], targets[s]
        )
        for s in SPLIT_ORDER
    )
    return work.drop(columns=['_tie_breaker']), current, final_objective


best_assignment = None
best_totals = None
best_objective = np.inf
attempt_results = []

for attempt in range(N_ATTEMPTS):
    assignment, totals, objective = allocate_groups(group_pd, RANDOM_SEED + attempt)
    attempt_results.append({'attempt': attempt + 1, 'objective': objective})
    if objective < best_objective:
        best_assignment = assignment
        best_totals = totals
        best_objective = objective

display(pd.DataFrame(attempt_results))
print('Best objective:', best_objective)

## 6. Join assignments back and validate the hard constraint

In [ ]:
assignment_schema = T.StructType([
    T.StructField(OVERLAP_GROUP_COL, T.StringType(), False),
    T.StructField(NEW_SPLIT_COL, T.StringType(), False),
])
assignment_for_spark = best_assignment[[OVERLAP_GROUP_COL, NEW_SPLIT_COL]].copy()
assignment_for_spark[OVERLAP_GROUP_COL] = assignment_for_spark[OVERLAP_GROUP_COL].astype('string')
assignment_for_spark[NEW_SPLIT_COL] = assignment_for_spark[NEW_SPLIT_COL].astype('string')
assignment_spk = spark.createDataFrame(
    assignment_for_spark,
    schema=assignment_schema,
)

features_with_new_split = features_grouped.join(
    assignment_spk, on=OVERLAP_GROUP_COL, how='left'
)

group_integrity = (
    features_with_new_split
    .groupBy(OVERLAP_GROUP_COL)
    .agg(F.countDistinct(NEW_SPLIT_COL).alias('number_of_new_splits'))
)
missing_assignments = features_with_new_split.filter(F.col(NEW_SPLIT_COL).isNull()).count()
groups_crossing_new_splits = group_integrity.filter(F.col('number_of_new_splits') != 1).count()

# Adjacent overlap components from the same pair must not overlap each other.
component_ranges = (
    features_grouped
    .groupBy(EMPLOYEE_COL, ACCOUNT_COL, 'overlap_component')
    .agg(
        F.min(WINDOW_START_COL).alias('component_start'),
        F.max(WINDOW_END_COL).alias('component_end'),
    )
)
component_order = Window.partitionBy(EMPLOYEE_COL, ACCOUNT_COL).orderBy('overlap_component')
overlapping_separate_components = (
    component_ranges
    .withColumn('previous_component_end', F.lag('component_end').over(component_order))
    .filter(F.col('component_start') <= F.col('previous_component_end'))
    .count()
)

hard_constraint_passed = (
    missing_assignments == 0
    and groups_crossing_new_splits == 0
    and overlapping_separate_components == 0
)
hard_constraint_summary = pd.DataFrame([{
    'missing_assignments': missing_assignments,
    'groups_crossing_new_splits': groups_crossing_new_splits,
    'overlapping_separate_components': overlapping_separate_components,
    'hard_constraint_passed': hard_constraint_passed,
}])
display(hard_constraint_summary)
assert hard_constraint_passed, 'Issue 3B overlap-group constraint failed'

## 7. Compare original and new label balance

In [ ]:
new_split_summary = (
    features_with_new_split
    .groupBy(NEW_SPLIT_COL)
    .agg(
        F.count('*').alias('new_row_count'),
        F.sum(TARGET_COL).alias('new_positive_count'),
        F.avg(TARGET_COL).alias('new_positive_rate'),
        F.countDistinct(OVERLAP_GROUP_COL).alias('new_group_count'),
    )
    .withColumn('new_row_share', F.col('new_row_count') / F.lit(total_rows))
    .withColumn('new_positive_share', F.col('new_positive_count') / F.lit(total_positives))
)

comparison_summary = (
    original_split_summary
    .select(
        F.col(ORIGINAL_SPLIT_COL).alias('split'),
        F.col('row_count').alias('original_row_count'),
        F.col('row_share').alias('original_row_share'),
        F.col('positive_count').alias('original_positive_count'),
        F.col('positive_share').alias('original_positive_share'),
        F.col('positive_rate').alias('original_positive_rate'),
    )
    .join(
        new_split_summary.select(
            F.col(NEW_SPLIT_COL).alias('split'),
            'new_row_count', 'new_row_share', 'new_positive_count',
            'new_positive_share', 'new_positive_rate', 'new_group_count',
        ),
        on='split',
    )
    .withColumn('row_share_difference', F.col('new_row_share') - F.col('original_row_share'))
    .withColumn('positive_share_difference', F.col('new_positive_share') - F.col('original_positive_share'))
    .withColumn('positive_rate_difference', F.col('new_positive_rate') - F.col('original_positive_rate'))
)
display(comparison_summary)

## 8. Simple feasibility decision

The thresholds match the issue 3A notebook and can be changed according to the project requirements.

In [ ]:
MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE = 0.02
MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE = 0.02
MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE = 0.10
MIN_POSITIVES_IN_VAL_OR_TEST = 100

comparison_pd = comparison_summary.toPandas()
comparison_pd['absolute_row_share_difference'] = comparison_pd['row_share_difference'].abs()
comparison_pd['absolute_positive_share_difference'] = comparison_pd['positive_share_difference'].abs()
comparison_pd['relative_positive_rate_difference'] = (
    comparison_pd['positive_rate_difference'].abs()
    / comparison_pd['original_positive_rate'].replace(0, np.nan)
)
comparison_pd['row_balance_passed'] = comparison_pd['absolute_row_share_difference'] <= MAX_ABSOLUTE_ROW_SHARE_DIFFERENCE
comparison_pd['positive_share_balance_passed'] = comparison_pd['absolute_positive_share_difference'] <= MAX_ABSOLUTE_POSITIVE_SHARE_DIFFERENCE
comparison_pd['positive_rate_balance_passed'] = comparison_pd['relative_positive_rate_difference'] <= MAX_RELATIVE_POSITIVE_RATE_DIFFERENCE
comparison_pd['minimum_positive_count_passed'] = True
comparison_pd.loc[
    comparison_pd['split'].isin(['val', 'test']),
    'minimum_positive_count_passed',
] = comparison_pd.loc[
    comparison_pd['split'].isin(['val', 'test']), 'new_positive_count'
] >= MIN_POSITIVES_IN_VAL_OR_TEST

all_balance_checks_passed = bool(comparison_pd[[
    'row_balance_passed', 'positive_share_balance_passed',
    'positive_rate_balance_passed', 'minimum_positive_count_passed',
]].all().all())
overall_feasible = bool(hard_constraint_passed and all_balance_checks_passed)

display(comparison_pd)
print('Hard constraint passed:', hard_constraint_passed)
print('All balance checks passed:', all_balance_checks_passed)
print('Overall issue 3B split considered feasible:', overall_feasible)

## 9. Optional save

The output is a sample-level assignment table. It can be joined back to the permanent feature table using employee, account, sample date, and window boundaries. The original feature table is never overwritten.

In [ ]:
split_assignment_output = (
    features_with_new_split
    .select(
        EMPLOYEE_COL, ACCOUNT_COL, SAMPLE_DATE_COL,
        WINDOW_START_COL, WINDOW_END_COL, TARGET_COL,
        ORIGINAL_SPLIT_COL, OVERLAP_GROUP_COL, NEW_SPLIT_COL,
    )
    .distinct()
)

if WRITE_OUTPUT:
    (
        split_assignment_output
        .write.format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .save(OUTPUT_PATH)
    )
    print('Saved:', OUTPUT_PATH)
else:
    print('WRITE_OUTPUT=False: assignment was generated and validated but not saved.')

## Interpretation

Issue 3B is feasible when the hard constraint passes and the new split remains within the configured balance thresholds. Compared with issue 3A, this approach permits non-overlapping time periods from the same employee-account pair to appear in different splits, which provides more allocation flexibility but weaker identity-level isolation.